# 🚀 JPX Tokyo Stock Exchange Prediction

## Ranking Model for Kaggle Competition

This notebook implements a comprehensive ranking model for the JPX Tokyo Stock Exchange Prediction competition.

---

### 📋 Table of Contents

1. [Installation & Setup](#installation--setup)
2. [Dependencies](#dependencies)
3. [Data Loading](#data-loading)
4. [Feature Engineering](#feature-engineering)
5. [Model Training](#model-training)
6. [Inference & Submission](#inference--submission)

---

## Installation & Setup

First, let's install the `jpx_ranker` package from GitHub.

> **Note on Dependency Warnings:** You may see dependency conflict warnings during installation (e.g., with protobuf, cudf, tensorflow, etc.). These are mostly harmless as they come from Kaggle's pre-installed packages that we don't use. The package will still work correctly. The critical dependencies for `jpx_ranker` (pandas, numpy, scikit-learn, lightgbm, xgboost, catboost, optuna) are properly constrained and compatible.

In [ ]:
!pip install git+https://github.com/AI-Ahmed/JPX_ranker.git

### 🔐 Kaggle Authentication

Configure Kaggle credentials to access competition data and datasets.

In [ ]:
from jpx_ranker.kaggle_setup import setup_kaggle_credentials, download_competition_data, download_dataset

# Setup credentials from .env file
setup_kaggle_credentials()
print("✓ Kaggle authentication configured")

### 📥 Download Competition Data

Download the required competition data and additional datasets.

> **Note:** This cell downloads data from Kaggle. Run it once, then you can delete it if desired.

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

# Download competition data
jpx_tokyo_stock_exchange_prediction_path = download_competition_data('jpx-tokyo-stock-exchange-prediction')

# Download additional datasets
dsxavier_jpx_pre_path = download_dataset('dsxavier/jpx-pre')
dsxavier_jpx_deps_v1_path = download_dataset('dsxavier/jpx-deps-v1')

print('Data source import complete.')



---

## 📦 Dependencies

Install all required packages and dependencies for the project.

## Install Dependencies

In [ ]:
!nvidia-smi

In [ ]:
!pip install -qq optuna catboost skfolio loguru

In [ ]:
dps_path = '/kaggle/input/jpx-deps-v1'
# deps
# - pandarallel
# - cudf-cu12 (Pandas Accelerator)

!pip install -qq --no-index --find-links {dps_path}/jpx-deps -r {dps_path}/requirements.txt
# get_ipython().kernel.do_shutdown(restart=True) # Kaggle disable ONLY

In [ ]:
%config Completer.use_jedi = False

## Import Dependencies

In [ ]:
# %load_ext cudf.pandas # Kaggle disable ONLY
import os
import warnings
import multiprocessing as mp
from pathlib import Path
from typing import Optional, Union, Tuple, List, Any
from collections import defaultdict

import pandas as pd
import numpy as np


from pandarallel import pandarallel as ppl
ppl.initialize(progress_bar=False, nb_workers=mp.cpu_count() - 1)

from tqdm.notebook import tqdm
from IPython.display import clear_output, display

from statsmodels.tsa.stattools import adfuller
from sklearn.base import BaseEstimator
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_absolute_error, mean_absolute_percentage_error
from skfolio.model_selection import CombinatorialPurgedCV

import optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

import matplotlib.pylab as plt
import seaborn as sns

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

os.environ['PYTHONHASHSEED'] = str(42)
np.random.seed(42)

# Data Preprocessing & Analysis

## Data Ingestion

In [ ]:
stk_prc = pd.read_csv('kaggle/input/jpx-tokyo-stock-exchange-prediction/train_files/stock_prices.csv')
stk_prc

In [ ]:
stk_prc = stk_prc.drop(columns=['RowId', 'AdjustmentFactor', 'ExpectedDividend', 'SupervisionFlag'])
stk_prc

In [ ]:
df = stk_prc.copy(deep=True)

## Data Preprocessing & Analysis

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df = df[['Date', 'SecuritiesCode', 'Close', 'Volume', 'Target']].dropna().reset_index(drop=True)
df

In [ ]:
tmp_df = df[df['SecuritiesCode'] == 4202]
tmp_df.Close.plot()

We must select events by comparing the shifted mean average to the target value using the cumulative buy/sell directions, subject to a threshold. To achieve this, we will employ an event-based sampling technique known as the CUMSUM Filter (Lopez de Parto, 2018). However, before implementing this method, it is essential to calculate both the volatility of each security and their average volatility.

In [ ]:
def get_daily_vol(close: pd.Series, span=100) -> pd.Series:
    """
    Compute the daily volatility of a time series.

    This function calculates the daily volatility of a time series using
    an exponentially weighted moving standard deviation (EWMSD) approach.

    Parameters
    ----------
    close : pd.Series
        Series containing the closing prices of the asset.
    span : int, optional
        Span parameter for the EWMSD calculation.
        Default is 100.

    Returns
    -------
    pd.Series
        Series containing the daily volatility.
    """
    # daily vol, reindexed to close
    df = close.index.searchsorted(close.index - pd.Timedelta(days=1))
    df = df[df > 0]
    df = pd.Series(close.index[df - 1], index=close.index[close.shape[0] - df.shape[0]:], name='sd').to_frame()
    df = (close.loc[df.index] / close.loc[df.sd].values) - 1 # daily return
    df = df.ewm(span=span).std()
    return df

In [ ]:
def get_events(df: pd.DataFrame, threshold: Union[int, float]=0.5) -> pd.DatetimeIndex:
    """
    Identify events in a time series based on a CUMSUM filter.

    This function detects events in a time series by applying a CUMSUM filter
    to a specified target column.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the time series data.
    threshold : Union[int, float], optional
        Threshold for detecting events.
        Default is 0.5.

    Returns
    -------
    pd.DatetimeIndex
        DatetimeIndex containing the timestamps of detected events.
    """
    events_t, s_pos, s_neg = [], 0, 0
    diff = df.Target
    for idx in diff.index:
        s_pos, s_neg = max(0, s_pos + diff.loc[idx]), min(0, s_neg + diff.loc[idx])

        if s_neg < -threshold:
            s_neg = 0; events_t.append(idx)
        elif s_pos > threshold:
            s_pos = 0; events_t.append(idx)

    return pd.DatetimeIndex(events_t)

Next, we aim to implement the triple-barrier method to categorize observations using predefined profit-taking and stop-loss thresholds. This approach is valuable for establishing profit and loss boundaries, as well as for determining return weights. Additionally, by incorporating this method, the model can identify securities that are expected to yield higher returns or experience reduced losses within a specified time frame.

In [ ]:
def apply_put_sell_on_threshold(events, close, put_sell) -> pd.DataFrame:
    r"""
    Apply put option selling strategy based on given threshold values.

    This function applies a put option selling strategy to events based on specified
    threshold values for profit taking (pt) and stop loss (sl).

    Parameters
    ----------
    events : pandas.DataFrame
        DataFrame containing event information, including ``tv1`` (end of event) and
        ``trgt`` (target value).
    close : pandas.Series
        Series containing closing prices.
        This series is used to calculate path prices and returns.
    put_sell : tuple
        Tuple containing threshold values for profit taking (pt) and stop loss (sl).
        The first element represents the profit taking threshold, and
        the second element represents the stop loss threshold.

    Returns
    -------
    pandas.DataFrame
        DataFrame containing the earliest profit taking ('pt') and
        stop loss ('sl') timestamps for each event.

    Notes
    -----
    The function calculates path returns based on the closing prices from the event
    start to the end.
    Profit taking (pt) and stop loss (sl) thresholds are applied to the path returns
    to determine the respective timestamps.
    """
    # apply stop loss/profit taking, if it takes place before tv1 (end of event)
    out = events[['tv1']].copy(deep=True)
    if put_sell[0] > 0:
        pt = put_sell[0] * events['trgt']
    else:
        pt = pd.Series(index=[events.name], dtype=np.float32) # `np.nan` values

    if put_sell[1] > 0:
        sl = -put_sell[1] * events['trgt']
    else:
        sl = pd.Series(index=[events.name], dtype=np.float32) # `np.nan` values

    loc_ = events.name
    tv1 = events['tv1']

    df = close[loc_: tv1] # path prices
    df = ((df / close[loc_]) - 1) * events['side'] # path returns

    if isinstance(pt, float):
        src_ = df[df > pt].tolist()
        out['pt'] = np.min(src_) if len(src_) > 0 else np.nan
    else:
        out['pt'] = df[df > pt[loc_]].index.min() # earliest profit taking

    if isinstance(sl, float):
        src_ = df[df < sl]
        out['sl'] = np.min(src_) if len(src_) > 0 else np.nan
    else:
        out['sl'] = df[df < sl[loc_]].index.min() # earliest stop loss.
    return out

In [ ]:
def get_vert_barrier(close: pd.Series,
                     t_events: pd.DatetimeIndex,
                     num_days: int) -> pd.Series:
    """
    Add a vertical barrier to events.

    Parameters
    ----------
    close : pd.Series
        Close prices of the security for given datetimes.
    t_events : pd.DatetimeIndex
        Timestamps representing event occurrences.
    num_days : int
        Number of days to define the vertical barrier.

    Returns
    -------
    pd.Series
        Timestamps indicating when the first vertical barrier is touched.

    Notes
    -----
    This function finds the timestamp of the next price bar at or immediately
    after a specified number of days from each event.
    The vertical barrier is defined as the time when the price reaches the specified
    number of days from the event timestamp.
    """
    tv1 = close.index.searchsorted(t_events + pd.Timedelta(days=num_days))
    tv1 = tv1[tv1 < close.shape[0]]
    tv1 = pd.Series(close.index[tv1], index=t_events[:tv1.shape[0]], name='tv1')  # NaNs at end
    return tv1

In [ ]:
def get_tbl_events(close: pd.Series,
                   t_events: pd.DatetimeIndex,
                   put_sell: Tuple[int, int],
                   trgt: pd.Series,
                   min_ret: float,
                   tv1: Union[bool, pd.Series] = False,
                   side: Optional[pd.Series] = None) -> pd.Series:
    r"""
    Generate a table of events based on specified parameters.

    Parameters
    ----------
    close : pd.Series
        Close prices of the security for given datetimes.
    t_events : pd.DatetimeIndex
        Timestamps representing event occurrences.
    put_sell : Tuple[int, int]
        Tuple containing threshold values for profit taking (pt) and stop loss (sl).
        The first element represents the profit taking threshold,
        and the second element represents the stop loss threshold.
    trgt : pd.Series
        Series containing target values.
        These values are used for filtering events based on minimum returns/vol.
    min_ret : float
        Minimum return/vol threshold for filtering events.
        Events with target values below this threshold are discarded.
    tv1 : Union[bool, pd.Series], optional
        Series containing the timestamp of the next price bar at or immediately
        after a number of days. If False, no vertical barrier is applied. Defaults to ``False``.
    side : Optional[pd.Series], optional
        Series containing side information for each event.
        This parameter is used to specify the side of each event (e.g., 1 for long, -1 for short).
        Defaults to ``None``.

    Returns
    -------
    pd.Series
        Table of events containing the timestamp of the vertical barrier (``tv1``),
        target values (``trgt``), profit taking timestamps (``pt``),
        and stop loss timestamps (``sl``).

    Notes
    -----
    Events are filtered based on minimum return thresholds and optionally by side information.
    The vertical barrier, profit taking, and stop loss are applied to each event according
    to the specified thresholds.
    """
    # 1) get targets
    com_idx = t_events.intersection(trgt.index)
    trgt = trgt.loc[com_idx] # sampling data based on CUM Filtering
    trgt = trgt.loc[trgt > min_ret] # filter by the min returns

    # 2) get tv1 (max holding period)
    if tv1 is False:
        tv1 = pd.Series(pd.NaT)

    # 3) form events object, apply stop loss on tv1
    if side is None:
        side_, pt_sl_ = pd.Series(1., index=trgt.index), [put_sell[0], put_sell[0]]
        events = pd.concat({'tv1': tv1, 'trgt': trgt,
                            'side': side_},
                        axis=1).dropna(subset=['trgt'])
    else:
        com_idx = side.index.intersection(trgt.index)
        side_, pt_sl_ = side.loc[com_idx], put_sell[:2]
        events = pd.concat({'tv1': tv1, 'trgt': trgt,
                            'side': side_},
                            axis=1).dropna(subset=['trgt', 'side'])

    events['tv1'] = events['tv1'].fillna(close.index[-1])

    df = events.parallel_apply(apply_put_sell_on_threshold, args=(close, pt_sl_), axis=1)
    events['pt'] = df['pt']
    events['sl'] = df['sl']
    events['tv1'] = df.dropna(how='all')['tv1']
    if side is None:
        events = events.drop('side', axis=1)

    return events

In [ ]:
def compute_tbl_events(df: pd.DataFrame,
                       put_sell: Tuple[int, int],
                       num_days: int = 1,
) -> pd.DataFrame:
    r"""
    Compute Triple Barriers events for each security in a DataFrame.

    This function computes Triple Barrier Labeling (TBL) events for each security
    in the provided DataFrame. It iterates over each unique ``SecuritiesCode`` and
    calculates TBL events based on the given parameters.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the time series data for multiple securities.
    put_sell : Tuple[int, int]
        Tuple containing the multipliers for the lower and upper barriers, respectively.
    num_days : int, optional
        Number of days to wait before closing the position, by default ``1``.

    Returns
    -------
    Tuple[pd.DataFrame, pd.Series]
            - pd.DataFrame: DataFrame containing the TBL events.
    """

    df_ = pd.DataFrame()
    lst_secs = df.SecuritiesCode.unique().tolist()
    prg_bar = tqdm(enumerate(lst_secs), total=len(lst_secs),
                   desc=f'Compute Triple barriers for {num_days} position')
    for idx, sec in prg_bar:
        sec_df = df[df['SecuritiesCode'] == sec].reset_index(drop=True)
        sec_df['Date'] = pd.to_datetime(sec_df['Date'])
        sec_df = sec_df.set_index('Date')

        vol = get_daily_vol(sec_df.Close)
        avg_vol = vol.mean()

        t_events = get_events(sec_df, avg_vol)
        t1 = get_vert_barrier(sec_df.Close, t_events, num_days)
        print(f"sec: {sec}, barrier len: {t1.shape[0]}")
        if t1.shape[0] > 50:
            events = get_tbl_events(sec_df.Close, t_events, put_sell,
                                    vol, avg_vol, t1)
            sec_df = sec_df.loc[events.index]
            sec_df = sec_df.reset_index().dropna()
            df_ = pd.concat([df_, sec_df])

        if idx % 5 == 0:
            clear_output(wait=True)
            display(prg_bar.container)
    df_ = df_.sort_values(by='index').reset_index(drop=True)
    df_.columns = df_.columns[1:].insert(0, 'Date')
    return df_

Prior to computing the triple barrier, we had already stored the outcomes in a private dataset because of issues encountered during parallel computation with Kaggle. Consequently, to save time, we executed the code on Colab, where it completed processing within 1 hour and 7 minutes, utilizing only one processor. Consequently, I determined that it would be more efficient to store the results in a dataset rather than recompute them during debugging and notebook commits.

In [ ]:
put_sell = (1, 1)
one_day = 1
two_days = 2
df_1, df_2 = None, None
data_path = Path('../kaggle/input/jpx-pre/versions/2/')
os.makedirs(data_path, exist_ok=True)
file_name = 'tbm_data.csv'

if os.path.exists(data_path / file_name):
    df = pd.read_csv(data_path / file_name)
else:
    df_1 = compute_tbl_events(df, put_sell, one_day)
    df_2 = compute_tbl_events(df, put_sell, two_days)
    df = pd.concat([df_1, df_2])
    df = df[~df.duplicated(keep='first')]
    df = df.dropna().reset_index(drop=True)
    df.to_csv(data_path / file_name, index=False)
df

Review the securities and eliminate those with fewer than 50 samples or 1.9E-4 percent. This strategy effectively removes the rarest samples, thereby preventing them from unduly influencing the model weights.

In [ ]:
df.SecuritiesCode.value_counts()

In [ ]:
df.SecuritiesCode.value_counts(normalize=True)

In [ ]:
def drop_securities(events: pd.DataFrame, min_pct: float = 1.9e-4, max_drop_sec: int = 400):
    r"""
    Review securities and eliminate those with fewer than a specified percentage of samples.

    This function reviews the securities in the provided DataFrame and
    eliminates those with fewer samples than the specified percentage.
    It iterates until all securities meet the minimum sample percentage criterion.

    Parameters
    ----------
    events : pd.DataFrame
        DataFrame containing the securities data.
    min_pct : float, optional
        Minimum percentage of samples allowed for each security, by default ``1.9e-4``.
    max_drop_sec : int, optional
        Maximum number of securities to drop, by default ``400``.
        We can't go less to fit the ranking system of the competition.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the remaining securities after elimination
        based on sample percentage.
    """
    # apply weights, drop labels with insufficient examples
    events_ = events
    while True:
        df = events_['SecuritiesCode'].value_counts(normalize=True)
        if df.min() > min_pct or df.shape[0] < max_drop_sec : break
        print(f'Dropped Label ({df.idxmin()}, {df.min()})')
        events_ = events_[events_['SecuritiesCode'] != df.idxmin()]

    return events_

In [ ]:
df = drop_securities(df)

In [ ]:
df.SecuritiesCode.value_counts(normalize=True)

Now that we have calculated the triple barrier events for each security, we need to recalculate the target returns to account for the alterations in the events we have identified.

In [ ]:
def cal_target(df):
    """
    Calculate target returns based on given periods (1-day return, 2-day returns).

    This function calculates target returns for each security in the provided DataFrame.
    It computes the percentage change in the closing price between the current day and the next day,
    and between the current day and the day after next, representing 1-day and 2-day returns respectively.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the time series data for multiple securities.

    Returns
    -------
    pd.DataFrame
        DataFrame containing the calculated target returns for each security.
    """
    df_ = pd.DataFrame()
    lst_secs = df.SecuritiesCode.unique().tolist()
    prg_bar = tqdm(enumerate(lst_secs), total=len(lst_secs),
                   desc=f'Compute Target Returns')
    for _, sec in prg_bar:
        sec_df = df[df['SecuritiesCode'] == sec].reset_index(drop=True)
        sec_df['Date'] = pd.to_datetime(sec_df['Date'])
        sec_df = sec_df.set_index('Date')

        # ref: https://www.kaggle.com/code/chumajin/easy-to-understand-the-competition?scriptVersionId=94143164&cellId=17
        sec_df["Close_shift1"] = sec_df["Close"].shift(-1)
        sec_df["Close_shift2"] = sec_df["Close"].shift(-2)
        sec_df["Target"] = (sec_df["Close_shift2"] - sec_df["Close_shift1"]) / sec_df["Close_shift1"]
        sec_df = sec_df[['SecuritiesCode', 'Close', 'Volume', 'Target']]

        sec_df = sec_df.dropna().reset_index()
        df_ = pd.concat([df_, sec_df])
    df_ = df_.sort_values(by='Date').reset_index(drop=True)
    return df_

In [ ]:
df = cal_target(df)
df

Let's have the volume weighted average price (VWAP) of each equity

In [ ]:
from _src.utils import compute_vwap

In [ ]:
df = compute_vwap(df)
df

In [ ]:
df[df.SecuritiesCode == 4202]

In [ ]:
tmp_df = df[df['SecuritiesCode'] == 4202]
plt.plot(tmp_df['Close'])
plt.show()

In [ ]:
plt.plot(tmp_df['VWAP'])
plt.show()

After comparing the charts for the selected security before and after applying our preprocessing steps, we observe a significant change in distribution. This improved distribution, achieved through our preprocessing, effectively targets long and short positions for entering and exiting the market for each security. Consequently, training a model to rank these positions becomes critical for guiding investor decisions and determining which securities to invest in or take a short position on, thereby refining the model's decision boundary and enhancing precision.

Now, let's conduct a stationarity test on both the `Target` and `rate_vwap` variables to determine which one to utilize in our subsequent processes.


In [ ]:
stats = adfuller(tmp_df.Target)
t_test, p_value = stats[:2]
t_test, p_value

In [ ]:
stats = adfuller(tmp_df.VWAP)
t_test, p_value = stats[:2]
t_test, p_value

## Feature Engineering

In [ ]:
from _src.utils import compute_feature_eng

In [ ]:
fast_period = 5
slow_period = 10

df = compute_feature_eng(df, fast_period, slow_period)
df

In [ ]:
df = df[['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']]
df['side'] = df['side'].astype(int)

# Compute correlation across all data (not grouped by SecuritiesCode and Date)
# since groupby().corr() on those columns leaves no variation within groups
df_vis = df[['Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']].corr()

plt.figure(figsize=(15, 10))
sns.heatmap(df_vis, annot=True, fmt='.2f')
plt.show()

In [ ]:
correlation_df = df.groupby('SecuritiesCode').corr().droplevel(0)
correlation_df

In [ ]:
plt.figure(figsize=(15, 10))
plt.imshow(correlation_df.values, cmap='viridis', aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(ticks=[])
plt.yticks(ticks=[])
plt.title('Correlation Heatmap')
plt.xlabel('Features')
plt.ylabel('Features')
plt.show()

In [ ]:
df = df[['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']]
df['Rank'] = df.groupby("Date")["Target"].rank(ascending=False,method="first") - 1
df['Rank'] = df['Rank'].astype('int')
df

In [ ]:
df.info()

In [ ]:
df = df.reset_index(drop=True)
os.makedirs('../kaggle/working/', exist_ok=True)
df.to_feather('../kaggle/working/preped_data.feather')

---

## 📊 Data Preparation & Modeling

Prepare the processed data and train ranking models for stock prediction.

In [ ]:
from jpx_ranker._src.utils import compute_vwap
from jpx_ranker._src.utils import compute_feature_eng

In [ ]:
df = pd.read_feather('kaggle/working/preped_data.feather')
df

In [ ]:
df_test = pd.read_csv('kaggle/input/jpx-tokyo-stock-exchange-prediction/supplemental_files/stock_prices.csv')
df_test

In [ ]:
fast_period = 5
slow_period = 10

# Check if the test preprocessing data is already exists previously:
if os.path.exists('kaggle/working/test_preped_data.feather'):
    df_fet_test = pd.read_feather('kaggle/working/test_preped_data.feather')
else:
    df_fet_test = df_fet_test[['Date', 'SecuritiesCode', 'Close', 'Volume', 'Target']].dropna().reset_index(drop=True)
    df_fet_test = compute_vwap(df_fet_test)
    df_fet_test = compute_feature_eng(df_fet_test, fast_period, slow_period)
    df_fet_test = df_fet_test[['Date', 'SecuritiesCode', 'Target', 'vwap_rate', 'side', 'vol5', 'vol10', 'vol15', 'vol21', 'autocorr-10', 'autocorr-15', 'autocorr-21']]
    df_fet_test['Rank'] = df_fet_test.groupby("Date")["Target"].rank(ascending=False,method="first") - 1
    df_fet_test['Rank'] = df_fet_test['Rank'].astype('int')
    df_fet_test.to_feather('../kaggle/working/test_preped_data.feather')

df_fet_test

After preprocessing our dataset and incorporating additional features, the next step is to prepare our dataset for training with a ranking model. The ranking model enables us to train models based on the extracted features and obtain the rank of each security's features. 

To avoid overfitting and data leakage in financial time series, we utilize an advanced cross-validation technique called **CombinatorialPurgedCV** from [skfolio](https://skfolio.org/generated/skfolio.model_selection.CombinatorialPurgedCV.html), which is specifically designed for financial machine learning. This approach offers significant advantages over standard `TimeSeriesSplit`:

- **Purging**: Removes training observations whose labels overlap in time with test labels, preventing leakage from multi-day return calculations
- **Embargoing**: Excludes observations immediately following test periods to handle serial correlation in financial features (ARMA processes, momentum effects)
- **Combinatorial Paths**: Creates multiple train/test combinations for more robust validation (e.g., C(10,2) = 45 splits instead of just 5)

Our implementation trains three state-of-the-art ranking models (`LGBMRanker`, `XGBRanker`, and `CatBoostRanker`) using **Optuna** for hyperparameter optimization. The best performing model is automatically selected based on validation NDCG@100 scores.

Additionally, we've developed a modular, professional pipeline architecture with:
- **DataProcessor**: Unified data preparation for all three frameworks
- **ModelSelector**: Automated model training, comparison, and selection
- **TrainingPipeline**: Complete training orchestration with evaluation
- **InferencePipeline**: Kaggle-compatible submission generation

This approach follows best practices from "Advances in Financial Machine Learning" by Marcos López de Prado, ensuring robust and production-ready algorithmic trading models.

In [ ]:
train = df.copy(deep=True)
test = df_fet_test.copy(deep=True)

In [ ]:
from jpx_ranker import calc_spread_return_sharpe
from jpx_ranker import calc_spread_return_sharpe_scorer

### 🔧 Data Processing

Initialize the `DataProcessor` to prepare data for model training.

In [ ]:
from jpx_ranker import DataProcessor

### 🎯 Model Training

Train multiple ranking models with Optuna hyperparameter optimization and select the best performer.

In [ ]:
from jpx_ranker import TrainingPipeline

In [ ]:
TRAIN_CONFIG = {
    'test_size': 0.2,          # Validation split ratio (20% for validation)
    'n_trials': 5, #200,           # Optuna trials per model (increase for better optimization)
    'n_folds': 5, #10,             # Number of folds for CombinatorialPurgedCV
    'n_test_folds': 2,         # Number of test folds per split
    'purged_size': 2,          # Days to purge around test sets (prevents label leakage)
    'embargo_size': 21,         # Days to embargo after test sets (handles serial correlation)
    'seed': 42,                # Random seed for reproducibility
    'save_model_path': 'kaggle/working/best_ranker_model.pkl',
    'device': 'cpu',           # Change to 'cuda' when submitting to Kaggle with GPU
    'min_trials': 5            # Minimum number of trials before checking for early stopping
}

# Feature engineering parameters (if needed)
UPDATED_FEATURE_CONFIG = {
    'fast_period': 5,
    'slow_period': 10
}

print("✓ Updated Configuration Loaded")
print(f"  Training config: {TRAIN_CONFIG}")
print(f"  Feature config: {UPDATED_FEATURE_CONFIG}")
print(f"\n  Note: Set n_trials=200 for production, n_trials=3 for quick testing")

In [ ]:
# Updated Training Pipeline with Fixed Model Names and Result Keys
# This cell uses the corrected configuration and properly accesses result dictionary keys

print("="*80)
print("STARTING MODEL TRAINING WITH UPDATED PIPELINE")
print("="*80)
print(f"Training data shape: {df.shape}")
print(f"Features: {df.columns.tolist()}\n")

# Initialize the training pipeline with updated configuration
pipeline = TrainingPipeline(
    test_size=TRAIN_CONFIG['test_size'],
    n_trials=TRAIN_CONFIG['n_trials'],
    n_folds=TRAIN_CONFIG['n_folds'],
    n_test_folds=TRAIN_CONFIG['n_test_folds'],
    purged_size=TRAIN_CONFIG['purged_size'],
    embargo_size=TRAIN_CONFIG['embargo_size'],
    seed=TRAIN_CONFIG['seed'],
    device=TRAIN_CONFIG['device'],
    min_trials=TRAIN_CONFIG['min_trials'],
)

print(f"Pipeline initialized with:")
print(f"  - Test size: {TRAIN_CONFIG['test_size']}")
print(f"  - Optuna trials per model: {TRAIN_CONFIG['n_trials']}")
print(f"  - CV folds: {TRAIN_CONFIG['n_folds']}")
print(f"  - Purge/Embargo: {TRAIN_CONFIG['purged_size']}/{TRAIN_CONFIG['embargo_size']} days")
print(f"  - Device: {TRAIN_CONFIG['device']}\n")

# Train the models (LightGBM, XGBoost, CatBoost)
print("Training models with Optuna hyperparameter optimization...")
pipeline.fit(df, save_path=TRAIN_CONFIG['save_model_path'])

# Display comprehensive results
print("\n" + "="*80)
print("FINAL TRAINING RESULTS")
print("="*80)
print(f"✓ Best Model: {pipeline.model_selector_.best_model_name_}")
print(f"✓ Best NDCG@100 Score: {pipeline.model_selector_.best_score_:.6f}")
print(f"✓ Training Sharpe Ratio: {pipeline.train_score_:.6f}")
print(f"✓ Validation Sharpe Ratio: {pipeline.val_score_:.6f}")

# Display detailed results for all models (using correct dictionary keys)
print("\n" + "-"*80)
print("DETAILED MODEL COMPARISON")
print("-"*80)
for model_name, results in pipeline.model_selector_.model_results_.items():
    print(f"\n{model_name}:")
    print(f"  ├─ NDCG@100:          {results['best_ndcg']:.6f}")
    print(f"  ├─ Best Sharpe:       {results['best_sharpe']:.6f}")
    print(f"  ├─ Trials Completed:  {results['n_trials_completed']}")
    print(f"  └─ Best Params:       {results['best_params']}")

print("\n" + "="*80)
print(f"✓ Model saved to: {TRAIN_CONFIG['save_model_path']}")
print("="*80)

---

## 🚀 Kaggle Submission

Generate predictions and submit to the Kaggle competition.

### Option 1: Automated Submission (Recommended)

Use the `run_kaggle_submission` function for a complete, production-ready submission workflow with automatic validation.

In [ ]:
from kaggle_submission import run_kaggle_submission

In [ ]:
run_kaggle_submission(
    model_path='kaggle/working/best_ranker_model.pkl',
    fast_period=5,
    slow_period=10,
    enable_submission=False  # Test mode
)


In [ ]:
from jpx_ranker import InferencePipeline
from jpx_ranker import calc_spread_return_sharpe

# The pipeline automatically extracts the best model and the matching data processor
inference_pipeline = InferencePipeline(model=pipeline) 
predictions = inference_pipeline.predict(df_test)
predictions

In [ ]:
eval_df = predictions[['Date', 'SecuritiesCode', 'Target', 'Rank', 'Score', 'PredictedRank']].sort_values(by=['Date', 'PredictedRank'])
eval_df = eval_df.drop(columns=['Rank'])
eval_df = eval_df.rename(columns={'PredictedRank': 'Rank'})
eval_df

In [ ]:
sharpe = calc_spread_return_sharpe(eval_df)
print(f"Test Sharpe Ratio: {sharpe:.4f}")